 Hai giai đoạn chính trong `rag.py`

### Giai đoạn A — Nạp tài liệu (ingestion)

Được thực hiện bởi hàm `add_document_to_rag()`.

```text
File tải lên
    ↓
read_file_text() — trích xuất văn bản
    ↓
RecursiveCharacterTextSplitter — chia văn bản thành chunk
    ↓
Document — đóng gói nội dung và metadata
    ↓
Embeddings — biểu diễn văn bản thành vector
    ↓
Chroma — lưu chunk, vector và metadata
```

Tài liệu thường được xử lý lúc người dùng tải lên, không phải đọc lại và chia lại từ đầu mỗi lần đặt câu hỏi.

### Giai đoạn B — Truy xuất khi người dùng hỏi

Được thực hiện bởi hàm `retrieve_from_rag()`.

```text
Câu hỏi người dùng
    ↓
Tạo embedding cho truy vấn
    ↓
Chroma similarity search
    ↓
Lọc theo thread_id và lấy tối đa k chunk
    ↓
Ghép nội dung cùng nhãn nguồn
    ↓
Trả về cho tool / agent
    ↓
LLM dùng nội dung tìm được để trả lời
```


- `Chroma`: lưu trữ và truy xuất vector.
- `GoogleGenerativeAIEmbeddings`: gọi mô hình embedding để biến văn bản thành vector.
- `Document`: đóng gói nội dung (`page_content`) và metadata.
- `RecursiveCharacterTextSplitter`: chia văn bản dài thành các chunk.
- `PdfReader`: trích xuất text từ PDF.
- `docx2txt`: trích xuất text từ DOCX.

### Khởi tạo embedding model và Chroma

```python
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

vectorstore = Chroma(
    collection_name="agentic_chatbot_docs",
    embedding_function=embeddings,
    persist_directory="chroma_db"
)
```

- `embeddings`: thành phần chuyển văn bản hoặc câu hỏi thành vector.
- `collection_name`: tên collection trong Chroma.
- `embedding_function`: quy định cách tạo embedding khi nạp dữ liệu và truy vấn.
- `persist_directory`: nơi Chroma lưu dữ liệu trên ổ đĩa để có thể dùng lại khi khởi động ứng dụng.






In [1]:
from pathlib import Path
from typing import List
import os
from dotenv import load_dotenv


load_dotenv()

True

In [2]:
Path("uploads").mkdir(exist_ok=True)
Path("chroma_db").mkdir(exist_ok=True)

In [ ]:
# Embedding model
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma



In [6]:
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-3-large",)

In [7]:
vectorstore = Chroma(
    collection_name="agentic_chatbot_docs",
    persist_directory="chroma_db",
    embedding_function=embedding_model)

In [9]:
from pypdf import PdfReader
from docx2txt import process as docx2txt

def read_file_txt(file_path: str) -> str:
    """Read a text file and return its content as a string."""
    path = Path(file_path)
    if not path.is_file():
        raise FileNotFoundError(f"The file {file_path} does not exist.")

    suffix = path.suffix.lower()
    if suffix == ".pdf":
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text

    if suffix == ".txt":
        return docx2txt.process(file_path)

    if suffix in [".txt", ".md", ".py", ".csv"]:
        return path.read_text(encoding="utf-8", errors="ignore")

    raise ValueError("Unsupported file type. Upload PDF, DOCX, TXT, MD, PY, or CSV.")


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def add_documents_to_rag(file_path: str, thread_id: str):
    txt = read_file_txt(file_path)

    # Split the text into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
    )

    chunks = text_splitter.split_text(txt)

    docs: List[Document] = [
        Document(page_content=chunk, metadata={"source": file_path, "thread_id": thread_id})
        for chunk in chunks
    ]

    vectorstore.add_documents(docs)
    return {
        "filename": Path(file_path).name,
        "chunks": len(docs)
    }






c:\Users\Admin\miniconda3\envs\tit\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
def retrieve_from_rag(query: str, thread_id: str, k: int = 4) -> str:
    docs = vectorstore.similarity_search(
        query,
        k=k,
        filter={"thread_id": thread_id}
    )

    if not docs:
        return "No relevant uploaded document content found."

    results = []

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "uploaded document")
        results.append(
            f"[Source {i}: {source}]\n{doc.page_content}"
        )

    return "\n\n".join(results)